# Natural Language Processing Lab 1
- Instructor: Luis Fernando Ramírez Ruiz
- Lead TA: Lonny Chen
- Published: September 2, 2026

# Objectives
Welcome to the first Natural Language Processing lab! 🙌🏼 🤓 🌟

Today's objectives are:

1. 🔎 **Explore** a research question together based on your text data!
2. 💡 Make the **concepts** from [Session 1](https://github.com/hertie-nlp-f2026/materials/blob/main/lectures/01_session-1/NLP_Session_1_Final.pdf) more tangible: NLP pipeline, choices affecting information loss, text representations.
3. 🛠️ Provide guidance for Python **environment setup** in Appendix @sec-python-setup.

## Research Question
Let's do some NLP on ... "NLP"! Specifically, our mini-research question is to identify clusters of topics and technical areas of interest by asking you to provide answers to:

>  **What do you want to learn about in this course and lab on Natural Language Processing?**

Learning about your various interests in NLP will help us customize future lab sessions. Analyzing open-ended questions like this one is also a common policy domain task, e.g., from a public opinion survey.

::: {.callout-warning title="Open-ended survey questions" collapse="true"}
Analyzing responses to open-ended survey questions can be challenging due to their often short length, variable spelling and grammar, and potentially multiple topics or themes per response. For the generated data in this lab, we also have a very small sample size.
:::

## Packages
Listed in this lab's `requirements.txt`. More details for updating packages are in the Appendix @sec-python-setup.

In [ ]:
#| code-summary: "Import packages"
#| output: false

# Data packages
import pandas as pd
pd.set_option('display.max_colwidth', 100) #default: 50 chars
pd.set_option('display.max_columns', None) #default: 20 columns
import numpy as np
from collections import Counter

# Plotting packages
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook'

# NLP packages
import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sentence_transformers import SentenceTransformer

# Text Input
Please submit **at least five responses** to the research question using the **Microsoft Form link provided** in the session slides. Try to vary them to cover various aspects of NLP: policy domain, technical concepts, applied models, etc. Your responses will be anonymously displayed for analysis.

We will save your responses in a CSV file, and then push it to this lab session's **GitHub folder**.

In the code below, we read the responses into a Pandas `DataFrame` and display them for review. Of course, the underlying data type is a Python **string**. A common term for each case of text data is a **document** so we name the column `doc` here for short.

In [ ]:
#| code-summary: "Load data to a `pandas` DataFrame"
data_file = 'nlp_lab1_data_in.csv'
df = pd.read_csv(data_file)
print(f'Data type: {df['doc'].dtype}')
print(f'Number of documents: {len(df)}')
df

# Model: Keyword to Topics
Let's use simple keyword search as a **rule or heuristic** to see which topics or themes come up in your responses.

::: {.callout-tip title="MODIFY"}
Modify the `topics` dictionary starting at **line 3** to create your own topics and keywords for each topic to search for.
:::

In [ ]:
#| code-summary: "Search for topics using keywords"
#| code-line-numbers: true
topics = {
    # MODIFY START: dictionary topic (key) and list of keywords (value)
    'policy': ['policy', 'government', 'administration'],
    
    # MODIFY END
}

topic_scores = pd.DataFrame([
    {
        topic: sum(word in doc.lower() for word in keywords) #search for substring
        for topic, keywords in topics.items() #per-topic-keywords
    }
    for doc in df['doc'] #per-doc
])

topic_scores.sum()

::: {.callout-tip title="DISCUSS"}
What information is lost when using this simple rule for analyzing topics?
:::

::: {.callout-note title="Searching for patterns" collapse="true"}
You'll learn about searching for more flexible patterns using "regular expressions" in **Session 2**.
:::

# Representation: Counts
## Preprocess to Word Tokens
For any downstream computational method, we need to **preprocess** the text strings. We will cover the steps of **tokenization** and **normalization** in detail in **Session 2**.

For now, it's enough to know that the strings need to be first split up into **tokens** (words here). The `spacy` library provides a processing pipeline that makes this efficient and parallel.

In [ ]:
#| code-summary: "Download `spacy` language model (once)"
#| output: false
spacy.cli.download('en_core_web_md')

::: {.callout-tip title="MODIFY"}
Modify the `rm_words` list at **line 4** to remove any words that you don't think would be useful in identifying topics or themes. Common **stopwords** such "a, an, I, we" are already being removed.
:::

In [ ]:
#| code-summary: "Preprocess text using `spacy` pipeline"
#| code-line-numbers: true
nlp = spacy.load('en_core_web_md', disable=['parser' ,'ner', 'textcat'])
docs_as_tokens = list(nlp.pipe(df['doc']))

rm_words = [] # MODIFY HERE: add to list of words to remove

docs_as_tokens_norm = [
    [t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop and not (t.lemma_.lower() in rm_words)]
    for doc in docs_as_tokens
]
pd.DataFrame({'tokens': docs_as_tokens_norm}).head()

## Convert to Count Vectors
Each document of tokens is now converted to  **count vectors** using the `scikit-learn` machine learning library. We will cover such doucment-token vectors in detail in **Session 3**. They are often used for downstream tasks such as document retrieval, traditional (non-neural) **Machine Learning** classification (**Session 5**) and **probability-based** topic models (**Session 7**).

In [ ]:
#| code-summary: "Convert to count vectors with `scikit-learn`"
docs_as_string_norm = [' '.join(doc) for doc in docs_as_tokens_norm]
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(docs_as_string_norm)
df_bow = pd.DataFrame(X_bow.toarray(), index=docs_as_string_norm, columns=vectorizer.get_feature_names_out())
print(f'The shape of the bag of words sparse matrix is {df_bow.shape}')
print(f'consisting of {df_bow.shape[0]} DOCUMENTS and {df_bow.shape[1]} tokens in its VOCABULARY')
df_bow.head(2)

::: {.callout-tip title="DISCUSS"}
What information is lost when converting to count vectors?
Do you see why these are called "sparse" vectors?
:::

## Visualize in Two Dimensions
We can visualize simplified differences in documents' count vectors by using a linear dimensionality reduction technique called [truncated Singular Value Decomposition (SVD)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html) from the `scikit-learn` library.

In [ ]:
#| code-summary: "Reduce count vectors to two dimensions and plot"
X_2d_bow = TruncatedSVD(n_components=2, random_state=42).fit_transform(X_bow)
df_X_2d_bow = pd.DataFrame(X_2d_bow, index=df['doc'], columns=['SVD1', 'SVD2'])

# Plot
fig = px.scatter(
    df_X_2d_bow.reset_index(names='doc'),
    x='SVD1', y='SVD2',
    hover_name='doc', hover_data={'SVD1': False, 'SVD2': False}
)
fig.update_layout(width=600, height=600)
fig.update_traces(marker_size=8)
fig.show()

::: {.callout-tip title="DISCUSS"}
Do you see any clusters of documents in this 2d scatter plot? Do they seem to have similar topics or themes? 
:::

# Representation: Embeddings
What if instead of counting words, we numerically represent the semantic "meaning" of each word? That is the power of **word embeddings** which we start to explore in **Session 4**.

## Static Embeddings
**Static** word embeddings are often used for both downstream **Machine Learning** and **Deep Learning** tasks such as classification which will be explored in **Sessions 5-6**. `spacy`'s language model also provides an **embedding vector** for each token in its vocabulary.

In [ ]:
#| code-summary: "Obtain and display embedding vectors for our vocabulary"
corpus_tokens = list({t for doc in docs_as_tokens_norm for t in doc})
df_we = pd.DataFrame(
    [nlp.vocab[t].vector for t in corpus_tokens],
    index=corpus_tokens
).sort_index().rename_axis('token')
df_we.head()

::: {.callout-tip title="DISCUSS"}
What information is lost when using static word embeddings?
Do you see why these are called "dense" vectors?
:::

We can also visualize simplified differences in word embeddings by using a linear dimensionality reduction technique, for example, [Principal component analysis (PCA)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) from the `scikit-learn` library.

::: {.callout-note title="Dimensionality reduction" collapse="true"}
Both SVD and PCA are "linear" dimensionality reduction techniques aimed at preserving as much dataset variation as possible while compressing the dimensionality of its case vectors. This compression is a linear transformation. "Non-linear" techniques exist, such as **UMAP**, which is generally better at preserving local clustering and is often used to reduce embedding dimensions themselves.
:::

In [ ]:
#| code-summary: "Reduce word embedding vectors to two dimensions and plot"
X_2d_wemb = PCA(n_components=2).fit_transform(df_we)
df_X_2d_wemb = pd.DataFrame(X_2d_wemb, index=df_we.index, columns=['PCA1', 'PCA2'])

# Plot
fig = px.scatter(
    df_X_2d_wemb.reset_index(names='token'),
    x='PCA1', y='PCA2',
    hover_name='token', hover_data={'PCA1': False, 'PCA2': False}
)
fig.update_layout(width=600, height=600)
fig.update_traces(marker_size=8)
fig.show()

::: {.callout-tip title="DISCUSS"}
Hover around the 2d space of word embeddings. Are similar words close together as expected? Are there are any proximities that are unexpected?
:::

`spacy` automatically generates **document embeddings** by **averaging** the document's word embeddings.

In [ ]:
#| code-summary: "Obtain and display document embedding vectors"
df_demb = pd.DataFrame(
    [doc.vector for doc in docs_as_tokens],
    index=[doc.text for doc in docs_as_tokens]
).rename_axis('doc')
df_demb.head(2)

In [ ]:
#| code-summary: "Reduce document embedding vectors to two dimensions and plot"
X_2d_demb = PCA(n_components=2).fit_transform(df_demb)
df_X_2d_demb = pd.DataFrame(X_2d_demb, index=df_demb.index, columns=['PCA1', 'PCA2'])

# Plot
fig = px.scatter(
    df_X_2d_demb.reset_index(names='doc'),
    x='PCA1', y='PCA2',
    hover_name='doc', hover_data={'PCA1': False, 'PCA2': False}
)
fig.update_layout(width=600, height=600)
fig.update_traces(marker_size=8)
fig.show()

::: {.callout-tip title="DISCUSS"}
Do you see any clusters of documents in this 2d scatter plot? Do they seem to more meaningfully clustered than with count vectors?
:::

## Contextual Embeddings
**Contextual** embeddings change based on the surrounding context in the document. Via the **HuggingFace ecosystem**, we download a small "sentence" embedding model called [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) that can be used with the `sentence-transformer` library. Other models can be found here: https://huggingface.co/sentence-transformers/models.

Contextual embeddings are used in **embeddings**-based topic models in **Session 7** and are essential for **Transformer**-based language models in **Sessions 8-9**.

::: {.callout-tip title="DISCUSS"}
Which of the **language levels** would contextual embeddings be aware of? Would it help to understand some of the **ambiguity** of language?
:::

::: {.callout-tip title="MODIFY"}
Download and use a different model, perhaps with more embedding dimensions. Modify **line 2** to use it and see if the PCA results below substantially change.
:::

In [ ]:
#| code-summary: "Download `sentence-transformer` embedding model"
#| code-line-numbers: true
sentence_model = SentenceTransformer('all-MiniLM-L6-v2') #384d (90MB)
#sentence_model = SentenceTransformer('modify_here') # MODIFY HERE

In [ ]:
docs_as_string = [doc.text for doc in docs_as_tokens]
X_demb_trf = sentence_model.encode(docs_as_string)
df_demb_trf = pd.DataFrame(X_demb_trf, index=docs_as_string).rename_axis('doc')
df_demb_trf.head(2)

In [ ]:
#| code-summary: "Reduce contextual document embedding vectors to two dimensions and plot"
X_2d_demb_trf = PCA(n_components=2).fit_transform(df_demb_trf)
df_X_2d_demb_trf = pd.DataFrame(X_2d_demb_trf, index=docs_as_string, columns=['PCA1', 'PCA2'])

# Plot
fig = px.scatter(
    df_X_2d_demb_trf.reset_index(names='doc'),
    x='PCA1', y='PCA2',
    hover_name='doc', hover_data={'PCA1': False, 'PCA2': False}
)
fig.update_layout(width=600, height=600)
fig.update_traces(marker_size=8)
fig.show()

::: {.callout-tip title="DISCUSS"}
Are the clusters of documents that you see here the most meaningful?
:::

# Evaluation
Although we didn't use our representations for a downstream task such as topic modeling, which choice would you make for text representation?

> To answer the **research question**: Do any topics or themes stand out to you through our analysis?

# Appendix: Python Setup {#sec-python-setup}
## Environment and Package Management
### Option A: Anaconda Distribution
**Anaconda** is probably the easiest "quick start" package that includes **Python**, an environment and package manager **conda**, and **Jupyter Notebooks** for running demonstration code as we will do in these labs. We will use Python 3.x. The current default version in Anaconda is **Python 3.13.15**.

- [Install Anaconda](https://www.anaconda.com/download)

After installation completes, go to **Environments** in the GUI and create a new environment for running the notebooks for these labs.

### Option B: Miniconda Distribution
**Miniconda** is a much smaller (< 1GB) version of Anaconda that can be downloaded from the same [website](https://www.anaconda.com/download/success). It provides Python, conda, and much few additonal packages. A comparison can be found [here](https://www.anaconda.com/docs/getting-started/concepts/anaconda-or-miniconda).

### Option X: Standalone Python and Other Package Managers
- [Install Python](https://www.python.org/downloads/)
- [Install UV](https://docs.astral.sh/uv/) (can also install Python)
- [Install Poetry](https://python-poetry.org/)

## Installing Packages
Each week's lab folder will contain a `requirements.txt` file that you can use to install the required packages and their versions. Depending on your environment, you can run:

- Anaconda ➡️ Environments (choose) ➡️ Open Terminal: `pip install -r requirements.txt`
- Add to a [UV](https://docs.astral.sh/uv/) project: `uv add -r requirements.txt`
- Temproary environments for ONLINE notebooks ([Google Colab](https://colab.research.google.com/), [Kaggle](https://www.kaggle.com/code)): `%pip install -r requirements.txt`

## Integrated Development Environment (IDE)
Although running labs and basic code development can be done in Jupyter Notebooks, a fully featured IDE is useful for larger projects, including your Research Note. Popular IDEs are currently:

- [Install VS Code](https://code.visualstudio.com/download) (Microsoft, most widely used)
- [Install Positron](https://positron.posit.co/) (makers of R Studio)
- [Install PyCharm](https://www.jetbrains.com/pycharm/) (free [Student Pack](https://www.jetbrains.com/academy/student-pack/) with school email)

## Lab-Ready Checklist

Course

- [ ] Install Anaconda distribution OR Python+package manager
- [ ] Create a new environment for these labs
- [ ] Install your preferred Python IDE

Per-Lab

- [ ] View `NLP_Lab_N.html` in your browser
- [ ] Install/update packages using `requirements.txt`  
- [ ] Open and run (import packages at least) `NLP_Lab_N_activity.ipynb` in your IDE or Jupyter